# 09 存活分析 — 練習

用松柏護理之家退伍軍人症資料練習存活分析。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

cases = df[df["infected"] == 1].copy()
cases["event"] = (cases["outcome"] == "dead").astype(int)

investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

## 題目 1：CHF（心衰竭）的存活分析

1. 畫出 CHF（`comorbidity_chf`）有 vs 無的 Kaplan-Meier 存活曲線
2. 執行 Log-rank 檢定
3. CHF 是否顯著影響存活？

In [ ]:
# TODO: CHF vs no CHF KM 曲線
# TODO: Log-rank 檢定
# TODO: 解讀

## 題目 2：年齡分組的存活比較

1. 將感染住民按年齡分成兩組：`age >= 75`（高齡）vs `age < 75`
2. 畫出兩組的 KM 存活曲線
3. 執行 Log-rank 檢定
4. 年齡是否顯著影響存活？

In [ ]:
# TODO: 分成 age >= 75 vs < 75
# TODO: KM 曲線
# TODO: Log-rank 檢定
# TODO: 解讀

## 題目 3（挑戰題）：住院 vs 未住院的存活分析 + Cox 迴歸

1. 畫出住院（`hospitalized == 1`）vs 未住院的 KM 存活曲線
2. 執行 Log-rank 檢定
3. 建立 Cox 迴歸模型，包含：`age`, `is_male`, `hospitalized`, `comorbidity_copd`, `comorbidity_chf`
4. 畫 HR 森林圖
5. 解讀：住院是「保護因子」還是「與嚴重度相關的標記」？

In [ ]:
# TODO: hospitalized vs not KM 曲線
# TODO: Log-rank 檢定
# TODO: Cox 迴歸（age, is_male, hospitalized, copd, chf）
# TODO: HR 森林圖
# TODO: 解讀

## 題目 4：結核病治療存活分析（TB 結核情境）

1. 使用 `drug_resistant`（是否為多重抗藥性結核 MDR-TB）將病人分成兩組
2. 畫出兩組達到痊癒（event）的 Kaplan-Meier 存活曲線
3. 執行 Log-rank 檢定，比較兩組痊癒時間是否有顯著差異
4. 分別計算兩組痊癒的中位治療天數（median survival time）
5. 解讀：抗藥性結核是否顯著延緩痊癒時間？對臨床照護與衛生政策有何意義？

In [ ]:
# 資料：結核病 (TB) 治療世代 -- 比較抗藥性 (MDR-TB) vs 非抗藥性病人達到痊癒的時間
rng = np.random.default_rng(409)
n = 400

drug_resistant = rng.binomial(1, 0.2, size=n)  # 20% 為多重抗藥性結核 (MDR-TB)
age = np.clip(rng.normal(48, 16, size=n), 15, 90).round().astype(int)

# 非抗藥性病人平均約 150 天痊癒；抗藥性病人療程長，平均約 320 天才痊癒
scale_cure = np.where(drug_resistant == 1, 320, 150)
duration_to_cure = rng.exponential(scale_cure)

follow_up_end = 540  # 追蹤 18 個月，超過此時間仍未痊癒者視為右設限（censored）
time_to_event = np.minimum(duration_to_cure, follow_up_end)
event = (duration_to_cure <= follow_up_end).astype(int)  # 1 = 痊癒, 0 = 追蹤結束仍未痊癒（設限）

tb = pd.DataFrame({
    "patient_id": [f"TB{i:04d}" for i in range(n)],
    "age": age,
    "drug_resistant": drug_resistant,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# TODO: 畫出 drug_resistant vs 非抗藥性病人的 KM 曲線（event = 痊癒）
# TODO: 執行 Log-rank 檢定
# TODO: 分別計算兩組痊癒的中位治療天數（median survival time）
# TODO: 解讀：抗藥性結核是否顯著延緩痊癒時間？對臨床照護與衛生政策有何意義？

## 題目 5：COVID-19 住院存活分析（ICU 分組）

1. 使用 `icu_admission` 將住院病人分成 ICU 與一般病房兩組
2. 畫出兩組住院期間存活（event = 死亡）的 Kaplan-Meier 曲線
3. 執行 Log-rank 檢定
4. 建立 Cox 迴歸模型，共變項包含：`age`, `is_male`, `icu_admission`, `diabetes`
5. 解讀：ICU 收治的風險比（HR）是否代表 ICU 治療本身提高死亡風險？還是反映病人本身病情較重（confounding by indication）？

In [ ]:
# 資料：COVID-19 住院世代 -- 比較 ICU 收治 vs 一般病房病人住院至死亡的時間
rng = np.random.default_rng(519)
n = 500

age = np.clip(rng.normal(60, 18, size=n), 18, 95).round().astype(int)
is_male = rng.binomial(1, 0.5, size=n)
icu_admission = rng.binomial(1, 0.22, size=n)
diabetes = rng.binomial(1, 0.25, size=n)

baseline_hazard = 1 / 420  # 一般病房、未患糖尿病、60 歲病人的基準死亡風險
linear_pred = 1.0 * icu_admission + 0.4 * diabetes + 0.03 * (age - 60)
hazard = baseline_hazard * np.exp(linear_pred)
duration = rng.exponential(1 / hazard)

follow_up_end = 60  # 60 天追蹤期，超過此時間仍住院或已出院者視為右設限
time_to_event = np.minimum(duration, follow_up_end)
event = (duration <= follow_up_end).astype(int)  # 1 = 住院期間死亡, 0 = 出院或追蹤結束（設限）

covid = pd.DataFrame({
    "patient_id": [f"CV{i:04d}" for i in range(n)],
    "age": age,
    "is_male": is_male,
    "icu_admission": icu_admission,
    "diabetes": diabetes,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# TODO: 畫出 ICU 收治 vs 一般病房病人的 KM 存活曲線（event = 住院期間死亡）
# TODO: 執行 Log-rank 檢定
# TODO: 建立 Cox 迴歸模型，共變項包含 age, is_male, icu_admission, diabetes
# TODO: 解讀 icu_admission 的風險比（HR）——這代表 ICU 治療本身有害，還是重症病人才會住 ICU？

## 題目 6：麻疹接觸者追蹤存活分析（measles 麻疹情境）

1. 使用 `vaccinated`（是否曾接種麻疹疫苗）將接觸者分成兩組
2. 畫出兩組在 21 天觀察期內發病（event）的 Kaplan-Meier 曲線
3. 執行 Log-rank 檢定
4. 計算未接種組的中位發病時間（可近似麻疹潛伏期中位數）
5. 解讀：接種疫苗是否顯著降低／延後發病？這對接觸者追蹤與檢疫期設定有何意義？

In [ ]:
# 資料：麻疹 (measles) 病例接觸者 -- 比較接種疫苗 vs 未接種疫苗接觸者暴露後發病的時間
rng = np.random.default_rng(626)
n = 350

vaccinated = rng.binomial(1, 0.6, size=n)  # 60% 接觸者曾接種麻疹疫苗
age = np.clip(rng.normal(10, 8, size=n), 0, 60).round().astype(int)

# 未接種者感染機率高（侵襲率高）；接種者多數有保護力，突破感染機率低
p_infected = np.where(vaccinated == 1, 0.12, 0.85)
infected = rng.binomial(1, p_infected)

# 若感染，潛伏期（暴露到發病）約 10-14 天；未感染者在觀察期內不會發病
incubation = np.clip(rng.normal(12, 2.2, size=n), 5, 21)
surveillance_end = 21  # 接觸者追蹤 21 天

time_to_event = np.where(infected == 1, incubation, surveillance_end)
event = infected  # 1 = 觀察期內發病, 0 = 觀察期結束仍未發病（設限）

measles = pd.DataFrame({
    "contact_id": [f"MS{i:04d}" for i in range(n)],
    "age": age,
    "vaccinated": vaccinated,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# TODO: 畫出接種 vs 未接種接觸者的 KM 曲線（event = 觀察期 21 天內發病）
# TODO: 執行 Log-rank 檢定
# TODO: 計算未接種組的中位發病時間（可近似麻疹潛伏期中位數）
# TODO: 解讀：接種疫苗是否顯著降低／延後發病？這對接觸者追蹤與檢疫期設定有何意義？

## 題目 7：登革熱重症化存活分析（dengue 登革熱情境）

1. 使用 `secondary_infection`（是否為次發感染）將病人分成兩組
2. 畫出兩組在 14 天臨床追蹤期內進展為重症登革熱（event）的 Kaplan-Meier 曲線
3. 執行 Log-rank 檢定
4. 建立 Cox 迴歸模型，共變項包含：`age`, `is_male`, `secondary_infection`
5. 解讀：次發感染的風險比（HR）是否顯著提高重症化風險？這是否符合抗體依賴增強效應（antibody-dependent enhancement, ADE）的機制？

In [ ]:
# 資料：登革熱 (dengue) 病例世代 -- 比較次發感染 vs 初次感染病人重症化的時間
rng = np.random.default_rng(727)
n = 450

secondary_infection = rng.binomial(1, 0.35, size=n)  # 35% 為次發感染（曾感染過不同型別登革病毒）
age = np.clip(rng.normal(32, 16, size=n), 1, 85).round().astype(int)
is_male = rng.binomial(1, 0.48, size=n)

baseline_hazard = 1 / 45  # 初次感染、32 歲病人的基準重症化風險
linear_pred = 1.1 * secondary_infection + 0.012 * (age - 32)
hazard = baseline_hazard * np.exp(linear_pred)
duration = rng.exponential(1 / hazard)

follow_up_end = 14  # 發病後 14 天臨床追蹤期
time_to_event = np.minimum(duration, follow_up_end)
event = (duration <= follow_up_end).astype(int)  # 1 = 進展為重症登革熱, 0 = 追蹤期結束仍未重症化（設限）

dengue = pd.DataFrame({
    "case_id": [f"DF{i:04d}" for i in range(n)],
    "age": age,
    "is_male": is_male,
    "secondary_infection": secondary_infection,
    "time_to_event": time_to_event.round(1),
    "event": event,
})

# TODO: 畫出次發感染 vs 初次感染病人的 KM 曲線（event = 進展為重症登革熱）
# TODO: 執行 Log-rank 檢定
# TODO: 建立 Cox 迴歸模型，共變項包含 age, is_male, secondary_infection
# TODO: 解讀：次發感染的風險比（HR）是否顯著提高重症化風險？這是否符合抗體依賴增強效應（ADE）的機制？

## 題目 8（挑戰題）：多重因子 Cox 迴歸與比例風險假設檢查（退伍軍人病情境）

延續本章開頭已載入的松柏護理之家 `cases`（感染病例，`event` = 死亡，`time_to_event` = 發病到死亡或追蹤結束的天數）。

1. 建立 Cox 迴歸模型，共變項包含：`age`, `is_male`, `icu_admission`, `immunosuppressed`, `comorbidity_cancer`
2. 印出每個變項的風險比（HR）與 95% 信賴區間
3. 使用 `cph.check_assumptions()` 檢查比例風險假設（proportional hazards assumption）是否成立
4. 計算模型的一致性指標（concordance index, C-index），評估模型的預測區辨能力
5. 解讀：哪些因子顯著提高死亡風險？比例風險假設是否違反？C-index 顯示模型的區辨能力如何？

In [ ]:
# 資料：延續本章開頭已載入的 cases（松柏護理之家退伍軍人症病例，event = 死亡）
cox_cols_q8 = ["time_to_event", "event", "age", "sex",
               "icu_admission", "immunosuppressed", "comorbidity_cancer"]
cox_df8 = cases[cox_cols_q8].copy()
cox_df8["is_male"] = (cox_df8["sex"] == "M").astype(int)
cox_df8 = cox_df8.drop(columns=["sex"])

print(cox_df8.describe().round(2))

# TODO: 建立 Cox 迴歸模型（age, is_male, icu_admission, immunosuppressed, comorbidity_cancer）
# TODO: 印出 HR 與 95% 信賴區間
# TODO: 用 cph.check_assumptions() 檢查比例風險假設
# TODO: 計算 C-index（cph.concordance_index_）
# TODO: 解讀